In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.medallion_data.ml_cache;
CREATE VOLUME IF NOT EXISTS workspace.medallion_data.mlflow_tmp;

In [0]:
import os

os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/medallion_data/ml_cache"


In [0]:
import mlflow
import mlflow.spark
import pandas as pd
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. Load Data from Gold Table
gold_df = spark.table("workspace.medallion_data.gold_telescope")
train_df, test_df = gold_df.randomSplit([0.8, 0.2], seed=42)

# 2. MANUALLY INDEX TARGET (The crucial fix!)
# Fit on train, transform both train and test.
indexer = StringIndexer(inputCol="class", outputCol="label", handleInvalid="skip").fit(train_df)
train_indexed = indexer.transform(train_df)
test_indexed = indexer.transform(test_df)

print("✅ Data loaded and target indexed. Ready for pipeline.")

In [0]:
# 3. BUILD THE DEPLOYMENT PIPELINE
bic_features = ["f_length", "f_size", "f_conc1", "f_m3long", "f_alpha"]

# Notice the pipeline now only contains the Assembler and LR model
assembler = VectorAssembler(inputCols=bic_features, outputCol="features", handleInvalid="skip")
lr = LogisticRegression(featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=[assembler, lr])

# 4. CROSS VALIDATION SETUP
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.01, 0.1])
             .addGrid(lr.elasticNetParam, [0.0, 0.5])
             .build())

evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5, 
    seed=42
)

print("✅ Pipeline and CrossValidator configured.")

In [0]:
# 5. TRAIN, EVALUATE, & LOG
with mlflow.start_run(run_name="telescope_pipeline_final") as run:
    current_run_id = run.info.run_id
    
    print("Training classic PySpark Pipeline across 5 folds...")
    
    # Train using the pre-indexed training data
    cv_model = cv.fit(train_indexed)
    best_pipeline = cv_model.bestModel
    
    # Evaluate using the pre-indexed test data
    test_predictions = best_pipeline.transform(test_indexed)
    
    best_cv_auc = max(cv_model.avgMetrics)
    test_auc = evaluator.evaluate(test_predictions)
    
    mc_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
    test_accuracy = mc_evaluator.evaluate(test_predictions, {mc_evaluator.metricName: "accuracy"})
    
    print("\n" + "="*30)
    print("🏆 METRICS COMPARISON")
    print("="*30)
    print(f"Cross-Validation AUC:  {best_cv_auc:.4f}")
    print(f"Holdout Test AUC:      {test_auc:.4f}")
    print(f"Holdout Test Accuracy: {test_accuracy:.4f}")
    print("="*30)

    # Fix the probability bug for the serverless endpoint
    best_pipeline.stages[-1].setProbabilityCol("")
    best_pipeline.stages[-1].setRawPredictionCol("")

    # Signature strictly maps the 5 expected inputs
    input_example = pd.DataFrame([{"f_length": 20.0, "f_size": 2.5, "f_conc1": 0.3, "f_m3long": 15.0, "f_alpha": 15.0}])
    output_example = pd.DataFrame({"prediction": [0.0]})
    signature = infer_signature(input_example, output_example)

    # Log model to MLflow (using your Volume cache!)
    mlflow.spark.log_model(
        spark_model=best_pipeline,
        artifact_path="telescope_model",
        signature=signature,
        input_example=input_example,
        dfs_tmpdir="/Volumes/workspace/medallion_data/ml_cache" 
    )
    
    print(f"\n✅ Training complete! Model logged to MLflow under Run ID: {current_run_id}")

In [0]:
# 6. REGISTER MODEL TO UNITY CATALOG
model_uri = f"runs:/{current_run_id}/telescope_model"
uc_model_name = "workspace.medallion_data.lr_telescope_model"

print(f"Registering model to Unity Catalog: {uc_model_name}...")

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=uc_model_name
)

print(f"✅ Successfully registered! Version: {registered_model.version}")
print(f"👉 Now go to the Databricks Serving UI and update your endpoint to Version {registered_model.version}!")

In [0]:
# 7. AUTOMATIC ENDPOINT DEPLOYMENT (Optional Cell 5)
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()

endpoint_name = "magic-telescope-endpoint"
uc_model_name = "workspace.medallion_data.lr_telescope_model"

# Automatically use the version we just registered in the previous cell!
model_version = registered_model.version 

print(f"Deploying {uc_model_name} (Version {model_version}) to endpoint '{endpoint_name}'...")

try:
    # Update existing endpoint to the new version
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[
            ServedEntityInput(
                entity_name=uc_model_name,
                entity_version=model_version,
                workload_size="Small",
                scale_to_zero_enabled=True
            )
        ]
    )
    print("✅ Endpoint update started! It will take ~5-10 minutes to become ready.")

except Exception as e:
    if "does not exist" in str(e).lower():
        print("⚠️ Endpoint doesn't exist yet. Creating it...")
        w.serving_endpoints.create(
            name=endpoint_name,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=uc_model_name,
                        entity_version=model_version,
                        workload_size="Small",
                        scale_to_zero_enabled=True
                    )
                ]
            )
        )
        print("✅ Endpoint creation started!")
    else:
        print(f"❌ Deployment failed: {e}")